<p align="center"> <img src="Data_Preperation.png" width="1000"/> </p>

##### The data preparation phase involved cleaning and structuring the raw dataset to ensure accuracy and consistency for analysis. Key steps included handling missing values, correcting data types, and creating important features such as YearMonth for time-based analysis and Revenue for financial insights. The data was also split into sales and returns based on quantity, enabling more focused and meaningful analysis of business performance.

<h2 style="color:white; background-color:#2E86C1; padding:10px; border-radius:5px;"> 🧹 Data Preparation  </h2>

1) Initial Data Overview
2) Fixing Incorrect DataTypes
3) Handling Missing Values
4) Remove Duplicates
5) Remove Invalid Business Record
6) Incorrect Inconsistent Formatting
7) Handling Outliers (IQR Method)
8) Create Revenue Column
9) Reset Index
10) Final Validation
11) Save Clean Dataset

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> Import Various Modules</h2> 

In [1]:
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import os

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> Creating Database Connection</h2> 

In [2]:
engine = create_engine("sqlite:///retail.db")

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> Load Raw Data</h2> 

In [3]:
df = pd.read_sql_query("SELECT * FROM online_retail;", engine)

original_rows = df.shape[0]
print("Original Shape:", df.shape)

Original Shape: (541909, 8)


In [4]:
#df_raw = pd.read_sql("SELECT * FROM online_retail;", engine)

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 1) Initial Data Overview</h2> 

In [5]:
df.info()
df.describe().round(2)

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   541909 non-null  int64  
 7   Country      541909 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 33.1+ MB


InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
dtype: int64

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 2) Fixing Incorrect DataTypes </h2> 

In [6]:
# Fixing Data Types

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['CustomerID'] = df['CustomerID'].astype('object')

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 3) Handling Missing Values</h2> 

In [7]:
df["Description"] = (
    df.groupby("StockCode")["Description"]
    .transform(lambda x: x.ffill().bfill())
)

df.shape
df.isna().sum()

InvoiceNo        0
StockCode        0
Description    112
Quantity         0
InvoiceDate      0
UnitPrice        0
CustomerID       0
Country          0
dtype: int64

In [32]:
df = df.dropna(subset=["Description"])
df.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
dtype: int64

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 4) Remove Duplicates </h2> 

In [11]:
# DROP DUPLICATES:

df = df.drop_duplicates(subset = ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice'])
duplicates = df.duplicated().sum()
print("Duplicate Rows:", duplicates)


Duplicate Rows: 0


<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 5) Remove Invalid Business Record </h2>

In [12]:
# Remove Cancelled Invoices

df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# Remove Invalid Business Record

df = df[~df["StockCode"].isin(["DOT", "D"])]

# Create a Clean Keyword Pattern

df = df[~df["Description"].str.contains(r"\?", na = False)]

# Check How Many Rows Match

#mask_adjustment = df["Description"].str.contains("|".join(pattern), case=False, na=False, regex=True)

# Remove

#df = df[~mask_adjustment]

In [13]:
# Use Regex 
# Check Validation junk data remove

df[df["Description"].str.contains(r"\?", na = False)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 6) Fix Inconsistent Formatting </h2>

### Upper/Lower Case

In [14]:
# Standardize:

df["Description"] = df["Description"].str.strip().str.upper()

df["StockCode"] = df["StockCode"].str.strip()

### Check Double Spaces Inside Text

In [15]:
# Fix

df["Description"] = df["Description"].str.replace(r"\s+", " ", regex=True)

### Check Spelling-Like Issues

In [16]:
df["Country"] = df["Country"].str.strip().str.title()

country_mapping = {
    "Usa": "United States",
    "Rsa": "South Africa",
    "Eire": "Ireland",
    "Unspecified": None,
    "European Community": None
}

df["Country"] = df["Country"].replace(country_mapping)

In [30]:
missing_pct = df["Country"].isna().mean() * 100
missing_pct

np.float64(0.10035547078203573)

In [31]:
df = df.dropna(subset=["Country"])

In [33]:
df.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
dtype: int64

In [18]:
# Keep real returnscorrect 
df = df[df["Quantity"] != 0]

df.shape

(526479, 8)

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 7) Handle Outliers (IQR Method) </h2>

In [19]:
# For Quantity

df_sales = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df_returns = df[df["Quantity"] < 0]

# Apply outlier removal ONLY on sales
Q1 = df_sales["Quantity"].quantile(0.25)
Q3 = df_sales["Quantity"].quantile(0.75)
IQR = Q3 - Q1

lower_q = Q1 - 1.5 * IQR
upper_q = Q3 + 1.5 * IQR

df_sales = df_sales[
    (df_sales["Quantity"] >= lower_q) &
    (df_sales["Quantity"] <= upper_q)
]


In [20]:
# For UnitPrice

Q1_p = df_sales["UnitPrice"].quantile(0.25)
Q3_p = df_sales["UnitPrice"].quantile(0.75)
IQR_p = Q3_p - Q1_p

lower_p = Q1_p - 1.5 * IQR_p
upper_p = Q3_p + 1.5 * IQR_p

df_sales = df_sales[
    (df_sales["UnitPrice"] >= lower_p) &
    (df_sales["UnitPrice"] <= upper_p)
]

# Remove only unrealistic extreme returns
df_returns = df_returns[df_returns["Quantity"] > -1000]

# Remove negative prices only
df_returns = df_returns[df_returns["UnitPrice"] >= 0]

# Combine back
df = pd.concat([df_sales, df_returns])

# Reset index
df = df.reset_index(drop=True)

In [21]:
print("Sales:", df[df["Quantity"] > 0].shape)
print("Returns:", df[df["Quantity"] < 0].shape)

Sales: (460241, 8)
Returns: (1119, 8)


In [22]:
df[df["Quantity"] < 0].sort_values("Quantity").head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
460323,540638,72038P,DAMAGES,-990,2011-01-10 12:14:00,0.0,15287,United Kingdom
460434,546023,85175,WRONGLY SOLD SETS,-975,2011-03-08 17:29:00,0.0,15287,United Kingdom
460317,540241,35957,SMALLFOLKART BAUBLE CHRISTMAS DEC,-939,2011-01-05 15:17:00,0.0,15287,United Kingdom
461204,573597,79341,"UNSALEABLE, DESTROYED.",-905,2011-10-31 15:18:00,0.0,15287,United Kingdom
461342,580382,16045,CHECK,-900,2011-12-02 17:59:00,0.0,15287,United Kingdom
461255,576368,21135,WET RUSTY,-864,2011-11-14 18:33:00,0.0,15287,United Kingdom
461174,572701,85078,MISSING,-840,2011-10-25 14:31:00,0.0,15287,United Kingdom
461099,569831,20713,WRONGLY CODED-23343,-800,2011-10-06 12:38:00,0.0,15287,United Kingdom
460718,553391,37370,SOLD AS SET/6 BY DOTCOM,-786,2011-05-16 16:42:00,0.0,15287,United Kingdom
461302,578328,72807B,WET PALLET,-696,2011-11-23 17:55:00,0.0,15287,United Kingdom


In [23]:
df[df["Quantity"] < 0].shape


(1119, 8)

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 8) Create Revenue Column </h2>

In [24]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 9) Reset Index </h2>

In [25]:
df = df.reset_index(drop=True)

<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 10) Final Validation </h2>

In [26]:
print("Final Shape:", df.shape)
print("Total Rows Removed:", original_rows - df.shape[0])
print("Sales:", (df["Quantity"] > 0).sum())
print("Returns:", (df["Quantity"] < 0).sum())
print("Negative Price:", (df["UnitPrice"] < 0).sum())
print("Remaining Cancelled Invoices:",
      df["InvoiceNo"].astype(str).str.startswith("C").sum())

Final Shape: (461360, 9)
Total Rows Removed: 80549
Sales: 460241
Returns: 1119
Negative Price: 0
Remaining Cancelled Invoices: 0


<h2 style="color:#1F618D; text-align:center; font-weight:bold;"> 11) Save Clean Dataset </h2>

In [27]:

df.to_csv("clean_retail_online_sales.csv", index = False)

In [28]:
df[df["Description"].str.contains(r"\?", na = False)].shape


(0, 9)

<p style="color: green; font-family:ComicSansMS; font-size: 20px;"> Final Inference

This project demonstrates a complete data wrangling pipeline...

- Cleaned missing values using group-based techniques  
- Removed duplicates and invalid transactions  
- Handled outliers using IQR  
- Created Revenue columns  

The dataset is now ready for analysis and visualization.